[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/02_synthetic_qa_generation.ipynb)

# Step 2 — Synthetic Q&A Generation Strategies

Compare how a **teacher LLM** generates policy Q&A using different prompting strategies.

## Learning objectives
- Zero-shot, one-shot, few-shot, and topic-controlled generation
- Understand distillation: strong model → synthetic training data → small model

In [1]:
from pathlib import Path

from aieng.syn_data.text import (
    PARAGRAPHS_PATH,
    SYNTHETIC_RAW_PATH,
    Paragraph,
    ParagraphSplit,
    compare_generation_strategies,
    create_teacher_client,
    generate_raw_synthetic_corpus,
    load_typed_jsonl,
    save_typed_jsonl,
    use_repo_root,
)
from dotenv import load_dotenv


load_dotenv()
use_repo_root(Path("."))

2026-06-11 14:44:47,274 INFO root: AI Engineering synthetic data utilities 

 Logging configured.


PosixPath('/Users/royajavadi/projects/synthetic-data-bootcamp')

In [2]:
# TODO: remove this before merging into main

%load_ext autoreload
%autoreload 2

## 1. Load TRAIN paragraphs only

Use paragraphs saved in Step 1 where `split == train`.

In [3]:
all_paragraphs = load_typed_jsonl(PARAGRAPHS_PATH, Paragraph.from_dict)
train_paragraphs = [p for p in all_paragraphs if p.split == ParagraphSplit.TRAIN]
print(f"Train paragraphs available: {len(train_paragraphs)}")
train_paragraphs[0]

Train paragraphs available: 7


Paragraph(doc_id='cfpb_credit_card_agreement', para_id='cfpb_credit_card_agreement::p0000', text='Sample Credit Card Agreement (excerpt for bootcamp scaffolding)', role=<DocumentRole.POLICY_DENSE: 'policy_dense'>, split=<ParagraphSplit.TRAIN: 'train'>, index=0)

## 2. Compare generation strategies on the same paragraph

In [ ]:
teacher = create_teacher_client()
demo_paragraph = train_paragraphs[0]

strategy_outputs = compare_generation_strategies(teacher, demo_paragraph)
for strategy, sample in strategy_outputs.items():
    print(f"\n=== {strategy} ===")
    print("Q:", sample.question)
    print("A:", sample.gold_answer)
    print("Failure mode:", sample.failure_mode)


=== zero_shot ===
Q: According to the provided text, what specific educational or training framework is the Sample Credit Card Agreement excerpt intended to support?
A: It is intended to be used as an excerpt for bootcamp scaffolding.
Failure mode: None

=== one_shot ===
Q: What is the purpose of the Sample Credit Card Agreement excerpt provided in the text?
A: The Sample Credit Card Agreement excerpt is provided for bootcamp scaffolding.
Failure mode: None

=== few_shot ===
Q: What is the designated purpose of the Sample Credit Card Agreement excerpt mentioned in the text?
A: The excerpt is intended to be used for bootcamp scaffolding.
Failure mode: None

=== topic_controlled ===
Q: According to the agreement, what requirement must be met to avoid paying interest on new purchases?
A: To avoid paying interest on new purchases, you must pay your entire balance (the New Balance) in full by the payment due date each month.
Failure mode: None


## Quick pick guide

Start simple → zero-shot 

Want consistent format → one/few-shot 

Want variety from long paragraphs → topic-controlled 

## 3. Save raw synthetic samples

Store outputs from each strategy for quality filtering in Step 3.

In [7]:
raw_samples = generate_raw_synthetic_corpus(
    teacher,
    train_paragraphs,
    max_paragraphs=min(5, len(train_paragraphs)),
)
print(f"Raw synthetic samples: {len(raw_samples)}")

save_typed_jsonl(
    SYNTHETIC_RAW_PATH,
    raw_samples,
    to_dict=lambda sample: sample.to_dict(),
)
SYNTHETIC_RAW_PATH

2026-06-11 15:17:33,557 INFO aieng.syn_data.text.generation: Payload: {'question': 'According to the provided text, what specific educational or training framework is the Sample Credit Card Agreement excerpt intended to support?', 'gold_answer': 'It is intended to be used as an excerpt for bootcamp scaffolding.'}
2026-06-11 15:17:36,635 INFO aieng.syn_data.text.generation: Payload: {'question': 'According to the provided text, what specific educational or training framework is the Sample Credit Card Agreement excerpt intended to support?', 'gold_answer': 'It is intended to be used as an excerpt for bootcamp scaffolding.'}
2026-06-11 15:17:59,361 INFO aieng.syn_data.text.generation: Payload: {'question': 'If a cardholder incurs a late payment in their second billing cycle and another in their fifth billing cycle, what APR may be applied to their account?', 'gold_answer': 'A Penalty APR of 29.99% may apply, as they have made two late payments within six billing cycles.'}
2026-06-11 15:18

Raw synthetic samples: 20


PosixPath('implementations/qa_text_generation/data/synthetic/synthetic_raw.jsonl')